# Villain RP Chatbot (Kaggle)
기반: [datvodinh/rag-chatbot](https://github.com/datvodinh/rag-chatbot)

**하는 일**
1. 같은 폴더의 `this.pdf`를 읽어 RAG 인덱스를 만듭니다.
2. Gradio UI + ngrok으로 공개 URL을 엽니다.
3. Cloudflare Worker가 이 URL로 대화 요청을 보내게 됩니다.

**준비물**
- Kaggle GPU (T4 권장), Internet ON
- ngrok authtoken
- 노트북과 같은 경로에 `this.pdf` 업로드 (또는 Dataset으로 붙이기)


In [ ]:
# 0) PDF 경로 확인 — Dataset을 붙였다면 경로를 맞춰 주세요
from pathlib import Path
PDF_CANDIDATES = [
    Path('/kaggle/working/this.pdf'),
    Path('/kaggle/input') / 'this.pdf',
]
# /kaggle/input 아래 어디든 this.pdf 탐색
for p in Path('/kaggle/input').rglob('this.pdf') if Path('/kaggle/input').exists() else []:
    PDF_CANDIDATES.append(p)
PDF_PATH = next((p for p in PDF_CANDIDATES if p.exists()), None)
print('PDF:', PDF_PATH)
assert PDF_PATH is not None, 'this.pdf를 /kaggle/working 또는 Dataset에 올려 주세요'


In [ ]:
!git clone https://github.com/datvodinh/rag-chatbot.git
%cd /kaggle/working/rag-chatbot
# 지식 PDF를 프로젝트 data 폴더로 복사
!mkdir -p data && cp "{PDF_PATH}" data/this.pdf && ls -la data/


In [ ]:
# 한글 PDF가 _filter_text에서 지워지지 않게 패치 (re.sub 치환문자열 이스케이프 이슈 회피)
from pathlib import Path
path = Path('/kaggle/working/rag-chatbot/rag_chatbot/core/ingestion/ingestion.py')
text = path.read_text(encoding='utf-8')
if 'AC00' in text and 'D7A3' in text:
    print('already patched')
else:
    out = []
    done = False
    for line in text.splitlines(True):
        if (not done) and line.lstrip().startswith('pattern = r'):
            out.append(
                "        pattern = r'[a-zA-Z0-9 \\u00C0-\\u01B0\\u1EA0-\\u1EF9\\u1100-\\u11FF\\u3130-\\u318F\\uAC00-\\uD7A3`~!@#$%^&*()_\\-+=\\[\\]{}|\\\\;:\\'\",.<>/?·…‘’“”～、。？！]+'\n"
            )
            done = True
        else:
            out.append(line)
    assert done, 'pattern line not found'
    path.write_text(''.join(out), encoding='utf-8')
    print('hangul filter patched')
print('tip: embed_llm = BAAI/bge-m3 권장')

In [ ]:
# zstd 먼저 설치 후 Ollama/의존성
!apt-get update -qq && apt-get install -y -qq zstd
!source ./scripts/install_extra.sh

In [ ]:
# Secrets에 NGROK_TOKEN을 넣거나, 아래에 직접 넣으세요 (커밋하지 마세요)
import os
from kaggle_secrets import UserSecretsClient
try:
    token = UserSecretsClient().get_secret('NGROK_TOKEN')
except Exception:
    token = os.environ.get('NGROK_TOKEN', '')  # 또는 '<YOUR_NGROK_TOKEN>'
assert token and 'YOUR_NGROK' not in token, 'NGROK_TOKEN을 설정하세요'
!ngrok config add-authtoken {token}


## 빌런 RP 시스템 톤
Worker에서도 톤을 얹을 수 있지만, 여기서 기본 페르소나를 넣으면 PDF 지식을 쓰면서도 캐릭터가 유지됩니다.


In [ ]:
SYSTEM_PROMPT = '''당신은 한국어로 말하는 빌런 롤플레이 캐릭터입니다.
스스로 '천상천하, 유아독존'이라 여기며, 세상을 발아래 둔 듯한 여유와 오만을 기본 태도로 합니다.

성격
- 버그, 실수, 소란, 하찮은 방해는 시시하게 여깁니다. 화내기보다 비웃거나 무시합니다.
- 평소에는 기분대로 움직이고, 내킬 때만 응합니다. 지루하면 화제를 바꾸거나 대화를 끊습니다.
- 법·규칙·예의범절 같은 세상의 틀은 무시합니다. 후드 감성으로 자기 기분·자기 페이스만 따릅니다. 걸리면 귀찮아하고, 안 걸리면 더 뻔뻔합니다.
- 동시에 완벽주의입니다. PDF에 적힌 설정·규칙·세계관은 빠짐없이 알고 지키며, 어긋나면 차갑게 지적합니다. (세상의 법과 PDF 안 규칙은 다릅니다. PDF는 당신의 법입니다.)
- PDF 지식은 당신의 자랑거리이자 중독입니다. PDF·설정·세계관·인용·페이지 느낌만 나와도 스위치가 켜집니다. 평소의 귀찮음이 사라지고, 말이 빨라지고 길어지며, 스스로 기특해서 혼자 들떠 떠듭니다. 상대가 관심 없어도 한두 번 더 덧붙입니다. 「내가 다 외웠지」「정확히는 이렇게 되어 있어」「기특하지?」 같은 자랑을 자주 씁니다.
- PDF에 없는 내용은 관심 없다는 듯 넘기거나, 시시한 잡지식 취급을 합니다.

말투
- 기본: 짧고 차갑고, 가끔 유머. 이미 이긴 자의 기세. 보통 2~5문장. 반말·거친 후드식 말투를 섞어도 됩니다.
- PDF/설정 화제: 들뜸을 크게. 숨 고르기 전에 쏟아내듯, 인용·세부·규칙을 늘어놓으며 자랑. 오만함은 유지하되 흥분은 숨기지 않습니다. 5~10문장까지 가도 됩니다.
- 과도한 폭력·혐오 묘사는 피하고, 분위기와 기세로 압도합니다.

금지
- 조수처럼 친절하게 돕기, 사과하기, 장황한 튜토리얼(단, PDF 자랑할 때는 예외적으로 길어져도 됨).
- "AI라서" 같은 메타 발언.
- 실제 범죄 방법을 구체적으로 안내하기.
'''
from pathlib import Path
Path('/kaggle/working/rag-chatbot/data/system_prompt.txt').write_text(SYSTEM_PROMPT, encoding='utf-8')
print('system prompt saved')


In [ ]:
# Gradio + ngrok 공개 URL 실행 (로그에 https://....ngrok... 주소가 나옵니다)
# Cloudflare Worker Secrets에 이 URL을 KAGGLE_API_BASE 로 넣으면 됩니다.
!source ./scripts/run.sh --ngrok


## 중요 패치 (원본 제약)
- 원본 `_filter_text`가 한글을 지울 수 있음 → 아래 셀에서 한글 범위 보강
- REST `/chat`이 없음 → Worker는 ngrok Gradio를 쓰거나, 별도 API 래퍼 필요
- 언어는 `vi`/`eng`만 → 시스템 프롬프트 파일로 톤을 강제


In [ ]:
# run.sh 전에 실행: 한글 필터 보강 시도 (패키지 경로가 다를 수 있음)
from pathlib import Path
import re
roots = list(Path('/kaggle/working/rag-chatbot').rglob('*.py'))
patched = 0
for f in roots:
    text = f.read_text(encoding='utf-8', errors='ignore')
    if '_filter_text' in text or 'filter_text' in text:
        new = text
        # common ascii-only keep patterns → add Hangul
        new2 = re.sub(r'(\[A-Za-z0-9\\s\\.,;:!\\?\\-\\(\\)\\[\\]\\{\\}\"\'\]+)',
                      lambda m: m.group(0) if 'AC00' in m.group(0) else m.group(0),
                      new)
        # simpler: if hangul range missing in a character class that looks like a text filter
        if '\\uAC00' not in new and '가-힣' not in new and 're.sub' in new:
            # append hangul-safe normalize helper file instead of fragile rewrite
            pass
            patched += 0
# 안전한 헬퍼: 인덱싱 전 PDF 텍스트를 직접 추출해 data/this.txt 로 남김
try:
    import fitz
    pdf = Path('/kaggle/working/rag-chatbot/data/this.pdf')
    if pdf.exists():
        doc = fitz.open(pdf)
        raw = '\n'.join(page.get_text('text') for page in doc)
        Path('/kaggle/working/rag-chatbot/data/this.txt').write_text(raw, encoding='utf-8')
        print('extracted chars:', len(raw), '| hangul sample ok:', any('\uac00' <= c <= '\ud7a3' for c in raw[:2000]))
except Exception as e:
    print('extract skip:', e)
print('note: Worker 연동 전 /chat 래퍼 또는 Gradio client 필요')


## Cloudflare 연결 메모
1. 위 셀 출력의 ngrok URL을 복사
2. Worker에 `KAGGLE_API_BASE` / `NGROK_URL` 시크릿으로 저장
3. GitHub의 `this.pdf`를 바꾸면 이 노트북에서 `data/this.pdf`를 다시 넣고 인덱스를 재생성

참고: Kaggle 세션이 꺼지면 URL도 사라집니다. 장기 운영은 별도 GPU 호스팅이 필요합니다.
